# Baseline experiments (Colab GPU)
Run on Colab only. No weights are downloaded with the default settings. Restart the runtime between configurations. See `docs/workflow.md` for packaging and limits.

In [ ]:
import os, subprocess, sys
assert 'google.colab' in sys.modules, 'Run this notebook in Google Colab'
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=True)
REPO = 'https://github.com/MichaelMukiibi/IEEE-CIS-DSQA-Starter-Kit.git'
REF = 'feat/colab-baseline-workflow'  # Replace with a committed SHA for repeat runs.
from pathlib import Path
ROOT = Path('/content/IEEE-CIS-DSQA-Starter-Kit')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPO, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run(['git', 'checkout', REF], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
# Explicit opt-ins. This model is a smoke-test substitute, not an official result.
ALLOW_MODEL_DOWNLOADS = False
RUN_INFERENCE = False
PREPARE_RAG_ASSETS = False
STRATEGY = 'zero_shot'  # or 'rag'; needs your own PDFs and auxiliary models
MODEL_ID = 'HuggingFaceTB/SmolLM2-135M-Instruct'
CORPUS_DIR = '/content/corpus'
LIMIT = 2
MAX_NEW_TOKENS = 128
# Do not select the 32B model unless you have explicitly provisioned sufficient GPU memory.
os.environ['HF_HUB_OFFLINE'] = '0' if ALLOW_MODEL_DOWNLOADS else '1'
os.environ['TRANSFORMERS_OFFLINE'] = os.environ['HF_HUB_OFFLINE']


In [ ]:
# uv manages Python dependencies. Install only direct baseline requirements;
# every version is read from the organizer's participant list.
import shutil
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', sys.executable, '.venv'], check=True)
PYTHON = str(ROOT / '.venv/bin/python')
names = {'torch', 'transformers', 'accelerate'}
if STRATEGY == 'rag':
    names.update({'numpy', 'docling', 'sentence-transformers'})
pins = [line.strip() for line in Path('requirements-participant.txt').read_text().splitlines()
        if '==' in line and line.split('==')[0] in names]
assert len(pins) == len(names)
subprocess.run(['uv', 'pip', 'install', '--python', PYTHON, *pins], check=True)
# Fail rather than silently replacing a pin that is unavailable.
subprocess.run(['uv', 'pip', 'check', '--python', PYTHON], check=True)
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


RAG uses user-supplied PDFs; the private evaluation corpus is unavailable. Asset preparation below explicitly runs the organizer build script, which replaces its local auxiliary model directories and downloads weights. Keep this disabled until intended. The script also creates the RAG submission zip.

In [ ]:
if PREPARE_RAG_ASSETS:
    assert ALLOW_MODEL_DOWNLOADS, 'Enable model downloads explicitly first'
    assert STRATEGY == 'rag'
    env = dict(os.environ, PATH=str(ROOT / '.venv/bin') + ':' + os.environ['PATH'])
    subprocess.run(['bash', '02_rag/build.sh'], env=env, check=True)
else:
    print('RAG asset downloads skipped')


In [ ]:
if RUN_INFERENCE:
    assert ALLOW_MODEL_DOWNLOADS, 'Explicitly enable weight loading/downloads first'
    # Execute reusable code in the uv environment; baseline modules stay unchanged.
    program = f"""
import torch
from experiments.runner import load_baseline, run
assert torch.cuda.is_available(), 'Select a Colab GPU runtime'
assert torch.cuda.is_bf16_supported(), 'The unchanged baselines require BF16 support'
module = load_baseline({STRATEGY!r})
module.MAX_NEW_TOKENS = {MAX_NEW_TOKENS!r}
result = run(module, model_id={MODEL_ID!r}, strategy={STRATEGY!r},
             corpus_dir={CORPUS_DIR!r}, limit={LIMIT!r}, output='results/run.json')
print(result['answers'])
"""
    subprocess.run([PYTHON, '-c', program], check=True)
else:
    print('Inference and generator weights skipped. Enable both opt-ins to run.')


In [ ]:
# Download recorded results before the Colab session ends.
if Path('results/run.json').exists():
    from google.colab import files
    files.download('results/run.json')
